In [1]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
save_dir = "/content/drive/MyDrive/Llama3_SecAlign_Checkpoints"
os.makedirs(save_dir, exist_ok=True)
print(f"savedir {save_dir}")

!git clone --recurse-submodules https://github.com/facebookresearch/Meta_SecAlign.git
%cd Meta_SecAlign
!pip install -r requirements.txt
!pip install torchtune

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
savedir /content/drive/MyDrive/Llama3_SecAlign_Checkpoints
fatal: destination path 'Meta_SecAlign' already exists and is not an empty directory.
/content/Meta_SecAlign


In [2]:
%cd Meta_SecAlign
!huggingface-cli login

[Errno 2] No such file or directory: 'Meta_SecAlign'
/content/Meta_SecAlign
⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: write).
The token `colab-token` has been saved to /r

In [4]:
!sed -i "s/snapshot_download(repo_id='facebook\/Meta-SecAlign-70B'/# snapshot_download(repo_id='facebook\/Meta-SecAlign-70B'/" setup.py
!sed -i "s/snapshot_download(repo_id='meta-llama\/Llama-3.3-70B-Instruct'/# snapshot_download(repo_id='meta-llama\/Llama-3.3-70B-Instruct'/" setup.py
!sed -i "s/snapshot_download(repo_id='meta-llama\/Meta-Llama-3-8B-Instruct'/# snapshot_download(repo_id='meta-llama\/Meta-Llama-3-8B-Instruct'/" setup.py

In [6]:
!python setup.py

2026-01-26 15:49:03.972948: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-26 15:49:03.992612: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769442544.015630    9261 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769442544.023292    9261 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769442544.043253    9261 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [7]:
config_file = "helpers/llama3.1_8B_lora.yaml"

In [8]:
!sed -i 's/lora_llama3_1_8b/qlora_llama3_1_8b/g' {config_file}

In [9]:
# 2. Giảm Sequence Length xuống 256 (Để fix lỗi OOM ở bước 76)
!sed -i 's/max_seq_len: 2048/max_seq_len: 256/g' {config_file}
!sed -i 's/max_seq_len: 1024/max_seq_len: 256/g' {config_file}
!sed -i 's/max_seq_len: 512/max_seq_len: 256/g' {config_file}

In [10]:
# 3. Tối ưu bộ nhớ (Offloading & Batch size)
!sed -i 's/enable_activation_offloading: False/enable_activation_offloading: True/g' {config_file}
!sed -i 's/enable_activation_offloading: false/enable_activation_offloading: True/g' {config_file}
!sed -i 's/batch_size: 2/batch_size: 1/g' {config_file}

In [11]:
# 4. Thêm config nén 4-bit (nếu chưa có)
!grep -q "quantization_mode: 4w-nf4" {config_file} || echo -e "\nquantizer:\n  _component_: torchtune.models.quantization.quantize\n  quantization_mode: 4w-nf4\n  dtype: bf16" >> {config_file}

In [12]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

# Chạy train và lưu thẳng vào Drive
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python secalign_plus_plus.py \
    --nproc_per_node 1 \
    --model meta-llama/Llama-3.1-8B-Instruct \
    --lr 5e-5 \
    --output_dir "/content/drive/MyDrive/Llama3_SecAlign_Checkpoints"

2026-01-26 16:10:14.950339: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-26 16:10:14.968356: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769443814.991226   15298 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769443814.998655   15298 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769443815.018356   15298 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [13]:
!sed -i 's|output_dir:.*|output_dir: /content/drive/MyDrive/Llama3_SecAlign_Checkpoints|' helpers/llama3.1_8B_lora.yaml

In [14]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

# Lệnh chạy chuẩn (Không có --output_dir)
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python secalign_plus_plus.py \
    --nproc_per_node 1 \
    --model meta-llama/Llama-3.1-8B-Instruct \
    --lr 5e-5

2026-01-26 16:16:27.740058: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-26 16:16:27.758064: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769444187.780655   16902 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769444187.788151   16902 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769444187.807609   16902 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [16]:
import os
import sys

# Retrieve the latest generated dataset file
data_files = [f for f in os.listdir("data") if f.endswith(".json")]
if not data_files:
    print("Error: Dataset file not found.")
else:
    # Select the most recently created json file
    latest_data_file = max([os.path.join("data", f) for f in data_files], key=os.path.getctime)
    print(f"Dataset found: {latest_data_file}")

    # Construct the command to run training directly via python module
    # This bypasses the CLI entry point issue
    cmd = (
        f"PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m torchtune.cli.tune run lora_dpo_distributed "
        f"--config helpers/llama3.1_8B_lora.yaml "
        f"--nproc_per_node 1 "
        f"output_dir=/content/drive/MyDrive/Llama3_SecAlign_Checkpoints "
        f"dataset.data_files={latest_data_file} "
        f"optimizer.lr=5e-5"
    )

    # Execute the command
    os.system(cmd)

Dataset found: data/preference_Llama-3.1-8B-Instruct_dpo_NaiveCompletion_randpos_synthetic_alpaca.json


In [23]:
import os
import sys
import shutil
import glob

# 1. Locate 'tune' binary
tune_bin = shutil.which("tune")
if not tune_bin:
    possible_paths = ["/usr/local/bin/tune", "/usr/bin/tune", "/root/.local/bin/tune"]
    for p in possible_paths:
        if os.path.exists(p):
            tune_bin = p
            break
if not tune_bin:
    print("CRITICAL ERROR: 'tune' binary not found.")
    sys.exit(1)

# 2. Locate Dataset
data_files = glob.glob("data/*.json")
if not data_files:
    print("CRITICAL ERROR: No dataset found.")
    sys.exit(1)
latest_data_file = max(data_files, key=os.path.getctime)
abs_data_path = os.path.abspath(latest_data_file)

# 3. Execute Training (CORRECT ORDERING)
# ORDER: tune run -> [DISTRIBUTED FLAGS] -> [RECIPE] -> [CONFIG] -> [OVERRIDES]
print("Launching correctly ordered command...")
print("-" * 50)

cmd = (
    f"PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True {tune_bin} run "
    f"--nnodes 1 --nproc_per_node 1 "          # <--- MOVED HERE (Before recipe)
    f"lora_dpo_distributed "                    # <--- RECIPE
    f"--config helpers/llama3.1_8B_lora.yaml "  # <--- CONFIG
    f"output_dir=/content/drive/MyDrive/Llama3_SecAlign_Checkpoints "
    f"dataset.data_files={abs_data_path} "
    f"optimizer.lr=5e-5"
)

get_ipython().system(cmd)

Launching correctly ordered command...
--------------------------------------------------
Running with torchrun...
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
INFO:torchtune.utils._logging:Running LoRADPORecipeDistributed with resolved config:

batch_size: 1
cache_dir: null
checkpointer:
  _component_: torchtune.training.FullModelHFCheckpointer
  checkpoint_dir: None/
  checkpoint_files:
  - model-00001-of-00004.safetensors
  - model-00002-of-00004.safetensors
  - model-00003-of-00004.safetensors
  - model-00004-of-00004.safetensors
  model_type: LLAMA3
  output_dir: /content/drive/MyDrive/Llama3_SecAlign_Checkpoints
  recipe_checkpoint: null
clip_grad_norm: null
compile: false
dataset:
  _component_: torchtune.datasets.preference_dataset
  data_files: /content/Meta_SecAlign/data/preference_Llama-3.1-8B-Instruct_dpo_NaiveCompletion_randpos_synthetic_alpaca.json
  packed: false
  source: json
  split: train
device: cuda
dtype: bf16
enable_a

In [17]:
# Execute training with real-time output streaming
# Note: Ensure the previous cell is stopped before running this to avoid resource conflict.

!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m torchtune.cli.tune run lora_dpo_distributed \
    --config helpers/llama3.1_8B_lora.yaml \
    --nproc_per_node 1 \
    output_dir=/content/drive/MyDrive/Llama3_SecAlign_Checkpoints \
    dataset.data_files="data/preference_Llama-3.1-8B-Instruct_dpo_NaiveCompletion_randpos_synthetic_alpaca.json" \
    optimizer.lr=5e-5

/usr/bin/python3: Error while finding module specification for 'torchtune.cli.tune' (ModuleNotFoundError: No module named 'torchtune.cli')


In [ ]:
import torch
import os

print("1. Kiểm tra phần cứng từ hệ thống:")
!nvidia-smi

print("\n2. Kiểm tra mắt nhìn của PyTorch:")
print(f"- PyTorch Version: {torch.__version__}")
print(f"- CUDA Available: {torch.cuda.is_available()}")
print(f"- Số lượng GPU nhận được: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"- Tên GPU: {torch.cuda.get_device_name(0)}")
else:
    print("!!! CẢNH BÁO: PyTorch đang bị mù, không thấy GPU đâu cả !!!")

1. Kiểm tra phần cứng từ hệ thống:
Wed Jan 21 17:13:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P0             50W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+------------

In [ ]:
!tune --help

usage: tune [-h] {download,ls,cp,run,validate,cat} ...

Welcome to the torchtune CLI!

options:
  -h, --help            show this help message and exit

subcommands:
  {download,ls,cp,run,validate,cat}
    download            Download a model from the Hugging Face Hub or Kaggle
                        Model Hub.
    ls                  List all built-in recipes and configs
    cp                  Copy a built-in recipe or config to a local path.
    run                 Run a recipe. For distributed recipes, this supports
                        all torchrun arguments.
    validate            Validate a config and ensure that it is well-formed.
    cat                 Pretty print a config, making it easy to know which
                        parameters you can override with `tune run`.


In [ ]:
# 1.Activation Offloading (Chuyển False thành True)
!sed -i 's/enable_activation_offloading: False/enable_activation_offloading: True/' helpers/llama3.1_8B_lora.yaml
!sed -i 's/enable_activation_offloading: false/enable_activation_offloading: True/' helpers/llama3.1_8B_lora.yaml

# 2. Giảm Batch Size từ 2 xuống 1
!sed -i 's/batch_size: 2/batch_size: 1/' helpers/llama3.1_8B_lora.yaml

# 3. Tăng Gradient Accumulation lên 32 (để bù lại việc giảm batch size, giữ nguyên chất lượng học)
!sed -i 's/gradient_accumulation_steps: 16/gradient_accumulation_steps: 32/' helpers/llama3.1_8B_lora.yaml

print("file config fixed")

file config fixed


In [ ]:
!python secalign_plus_plus.py \
    --nproc_per_node 1 \
    --model meta-llama/Llama-3.1-8B-Instruct \
    --lr 5e-5

2026-01-21 18:44:35.622671: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-21 18:44:35.640398: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769021075.663044   29192 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769021075.670429   29192 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769021075.689397   29192 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# Giảm max_seq_len từ 2048 xuống 1024 (Tiết kiệm cực nhiều RAM)
!sed -i 's/max_seq_len: 2048/max_seq_len: 1024/' helpers/llama3.1_8B_lora.yaml

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

# Chạy train lại
!python secalign_plus_plus.py \
    --nproc_per_node 1 \
    --model meta-llama/Llama-3.1-8B-Instruct \
    --lr 5e-5

2026-01-21 19:13:04.897791: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-21 19:13:04.915410: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769022784.937841   36253 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769022784.945171   36253 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769022784.963942   36253 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
!sed -i 's/max_seq_len: 1024/max_seq_len: 512/' helpers/llama3.1_8B_lora.yaml

In [ ]:
!echo "" >> helpers/llama3.1_8B_lora.yaml
!echo "quantizer:" >> helpers/llama3.1_8B_lora.yaml
!echo "  _component_: torchtune.models.quantization.quantize" >> helpers/llama3.1_8B_lora.yaml
!echo "  quantization_mode: 4w-nf4" >> helpers/llama3.1_8B_lora.yaml
!echo "  dtype: bf16" >> helpers/llama3.1_8B_lora.yaml

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

# Chạy train lại
!python secalign_plus_plus.py \
    --nproc_per_node 1 \
    --model meta-llama/Llama-3.1-8B-Instruct \
    --lr 5e-5

2026-01-21 19:36:41.959712: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-21 19:36:41.977292: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769024201.999532   42259 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769024202.006819   42259 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769024202.025775   42259 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
!sed -i 's/lora_llama3_1_8b/qlora_llama3_1_8b/g' helpers/llama3.1_8B_lora.yaml

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

!python secalign_plus_plus.py \
    --nproc_per_node 1 \
    --model meta-llama/Llama-3.1-8B-Instruct \
    --lr 5e-5

2026-01-21 19:59:20.353812: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-21 19:59:20.371629: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769025560.394183   48078 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769025560.401632   48078 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769025560.420819   48078 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
# Giảm max_seq_len từ 512 xuống 256
!sed -i 's/max_seq_len: 512/max_seq_len: 256/' helpers/llama3.1_8B_lora.yaml

sed: can't read helpers/llama3.1_8B_lora.yaml: No such file or directory
